In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import pandas as pd

df = pd.read_csv("data.csv")

print("Length :", len(df))
print("Shape :", df.shape)
print("Total Elements :", df.size)

Length : 4600
Shape : (4600, 18)
Total Elements : 82800


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           4600 non-null   object 
 1   price          4600 non-null   float64
 2   bedrooms       4600 non-null   float64
 3   bathrooms      4600 non-null   float64
 4   sqft_living    4600 non-null   int64  
 5   sqft_lot       4600 non-null   int64  
 6   floors         4600 non-null   float64
 7   waterfront     4600 non-null   int64  
 8   view           4600 non-null   int64  
 9   condition      4600 non-null   int64  
 10  sqft_above     4600 non-null   int64  
 11  sqft_basement  4600 non-null   int64  
 12  yr_built       4600 non-null   int64  
 13  yr_renovated   4600 non-null   int64  
 14  street         4600 non-null   object 
 15  city           4600 non-null   object 
 16  statezip       4600 non-null   object 
 17  country        4600 non-null   object 
dtypes: float

In [4]:
df.nunique()

date               70
price            1741
bedrooms           10
bathrooms          26
sqft_living       566
sqft_lot         3113
floors              6
waterfront          2
view                5
condition           5
sqft_above        511
sqft_basement     207
yr_built          115
yr_renovated       60
street           4525
city               44
statezip           77
country             1
dtype: int64

In [5]:
print(f"Dupilcate rows count : {df.duplicated().sum()}")

Dupilcate rows count : 0


In [6]:
df["date"] = pd.to_datetime(df["date"])

df["sale_year"] = df["date"].dt.year
df["sale_month"] = df["date"].dt.month

df = df.drop(columns=["date"])

In [7]:
df = df.drop(
    columns=[
        "street",
        "country",
        "sqft_above",
        "sqft_basement"
    ]
)

In [8]:
X = df.drop(columns=["price"])
y = df["price"]

In [9]:
numerical_columns = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = X.select_dtypes(
    include=["object"]
).columns.tolist()


print("\nNumerical Columns:")
print(numerical_columns)

print("\nCategorical Columns:")
print(categorical_columns)

print("\nPreprocessing preparation completed!")


Numerical Columns:
['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'yr_built', 'yr_renovated']

Categorical Columns:
['city', 'statezip']

Preprocessing preparation completed!


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)

Training Data: (3680, 14)
Testing Data: (920, 14)


In [11]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.impute import SimpleImputer

# Numerical preprocessing
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both pipelines
preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

# Fit on training data and transform
X_train_processed = preprocessor.fit_transform(X_train)

# Transform testing data using training parameters
X_test_processed = preprocessor.transform(X_test)

print("Training Shape:", X_train_processed.shape)
print("Testing Shape:", X_test_processed.shape)


# - `fit_transform()` learns preprocessing parameters from training data.
# - `transform()` applies those same parameters to testing data.

Training Shape: (3680, 130)
Testing Shape: (920, 130)


# Complete Regression Models – Machine Learning

**We will skip repeated preprocessing, train-test splitting, evaluation and visualization code. Below are only the model-specific implementations to insert after your existing preprocessing code.**

## Assumed preprocessed variables

```python
X_train_processed
X_test_processed
y_train
y_test
```

**Important:** Use the same train-test split for every model.

---

# 1. Linear Regression

**Idea:** Linear Regression predicts a continuous target by finding the best-fitting straight-line relationship between input features and the target variable.

It minimizes the sum of squared differences between actual and predicted values.

**Implementation:**

```python
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
```

**Important parameters:**

- `fit_intercept`: Whether to calculate the intercept.
- `copy_X`: Whether to copy the input data.
- `n_jobs`: Number of CPU cores used.
- `positive`: Forces coefficients to be positive.

**When to use:**

- When the relationship between features and target is approximately linear.
- As a baseline model for regression problems.
- When interpretability is important.

---

# 2. Polynomial Regression

**Idea:** Polynomial Regression captures nonlinear relationships between features and the target by creating polynomial combinations of the original features.

For example:

- Degree 1: Linear relationship.
- Degree 2: Includes squared features and pairwise interactions.
- Degree 3: Includes cubic terms.

**Implementation:**

```python
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

model = Pipeline([
    ("poly", PolynomialFeatures(
        degree=2,
        include_bias=False
    )),
    ("linear", LinearRegression())
])

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
```

**Important parameters:**

- `degree`: Degree of polynomial features.
- `include_bias`: Whether to include a constant column.
- `interaction_only`: Whether to generate only interaction features.
- `order`: Output array order.

**When to use:**

- When the relationship between features and target is nonlinear.
- When a straight-line model underfits the data.
- When polynomial patterns exist in the dataset.

**Important note:** Polynomial features can significantly increase the number of features. With a high-dimensional one-hot encoded dataset, use a small degree and monitor memory usage and overfitting.

---

# 3. Ridge Regression

**Idea:** Ridge Regression is Linear Regression with L2 regularization.

It adds a penalty to large coefficient values to reduce overfitting.

Unlike Lasso Regression, Ridge Regression generally does not make coefficients exactly zero.

**Implementation:**

```python
from sklearn.linear_model import Ridge

model = Ridge(
    alpha=1.0,
    solver="auto",
    max_iter=None,
    tol=0.0001
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
```

**Important parameters:**

- `alpha`: Controls the strength of regularization.
- `solver`: Algorithm used to calculate coefficients.
- `max_iter`: Maximum number of iterations.
- `tol`: Convergence tolerance.
- `fit_intercept`: Whether to calculate the intercept.

**When to use:**

- When there are many correlated features.
- When Linear Regression overfits.
- When you want to reduce large coefficient values.
- When working with high-dimensional datasets.

**Important note:** Ridge Regression works best when numerical features are scaled.

---

# 4. Lasso Regression

**Idea:** Lasso Regression uses L1 regularization. It shrinks coefficients and can make some coefficients exactly zero.

This helps with feature selection.

**Implementation:**

```python
from sklearn.linear_model import Lasso

model = Lasso(
    alpha=1.0,
    max_iter=10000
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
```

**Important parameters:**

- `alpha`: Regularization strength.
- `max_iter`: Maximum number of iterations.
- `tol`: Convergence tolerance.

**When to use:**

- When you want automatic feature selection.
- When your dataset contains irrelevant features.
- When you want a simpler model.

---

# 5. Elastic Net Regression

**Idea:** Elastic Net combines L1 and L2 regularization.

It combines the feature-selection capability of Lasso with the coefficient-shrinking capability of Ridge.

**Implementation:**

```python
from sklearn.linear_model import ElasticNet

model = ElasticNet(
    alpha=1.0,
    l1_ratio=0.5,
    max_iter=10000
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
```

**Important parameters:**

- `alpha`: Overall regularization strength.
- `l1_ratio`: Controls the balance between Lasso and Ridge.
- `max_iter`: Maximum number of iterations.
- `tol`: Convergence tolerance.

**When to use:**

- When there are many correlated features.
- When you need feature selection and regularization.
- When Lasso alone is unstable.

---

# 6. Decision Tree Regression

**Idea:** Decision Tree Regression splits data into smaller groups using feature conditions and predicts based on the values in each group.

It captures nonlinear relationships.

**Implementation:**

```python
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor(
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
```

**Important parameters:**

- `max_depth`: Maximum depth of the tree.
- `min_samples_split`: Minimum samples required to split a node.
- `min_samples_leaf`: Minimum samples required in a leaf.
- `criterion`: Function used to measure split quality.

**When to use:**

- When relationships are nonlinear.
- When you want an interpretable tree-based model.
- When feature scaling is not desirable.

---

# 7. Random Forest Regression

**Idea:** Random Forest combines multiple decision trees and averages their predictions.

It reduces the variance of individual decision trees and helps prevent overfitting.

**Implementation:**

```python
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
```

**Important parameters:**

- `n_estimators`: Number of trees.
- `max_depth`: Maximum depth of each tree.
- `max_features`: Number of features considered for each split.
- `min_samples_leaf`: Minimum samples in a leaf.
- `n_jobs`: Number of CPU cores.

**When to use:**

- When you have complex nonlinear relationships.
- When you want a powerful general-purpose regression model.
- When a single decision tree overfits.

---

# 8. Extra Trees Regression

**Idea:** Extra Trees builds multiple randomized decision trees and combines their predictions.

It introduces additional randomness during tree construction.

**Implementation:**

```python
from sklearn.ensemble import ExtraTreesRegressor

model = ExtraTreesRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
```

**Important parameters:**

- `n_estimators`: Number of trees.
- `max_depth`: Maximum tree depth.
- `min_samples_split`: Minimum samples needed to split.
- `max_features`: Number of features considered per split.

**When to use:**

- When you want to compare against Random Forest.
- When you need a randomized tree ensemble.
- When you want to capture nonlinear patterns.

---

# 9. AdaBoost Regression

**Idea:** AdaBoost trains models sequentially, giving more attention to observations that previous models predicted poorly.

It combines multiple weak learners into a stronger model.

**Implementation:**

```python
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor

base_model = DecisionTreeRegressor(
    max_depth=4
)

model = AdaBoostRegressor(
    estimator=base_model,
    n_estimators=100,
    learning_rate=0.05,
    random_state=42
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
```

**Important parameters:**

- `estimator`: Base learning algorithm.
- `n_estimators`: Number of boosting stages.
- `learning_rate`: Contribution of each learner.
- `loss`: Loss function used for updating weights.

**When to use:**

- When simple models underfit.
- When you want to combine multiple weak learners.
- When you want to experiment with boosting.

---

# 10. Gradient Boosting Regression

**Idea:** Gradient Boosting builds models sequentially, with each new model trying to correct the previous model's prediction errors.

It uses gradient descent principles to minimize a loss function.

**Implementation:**

```python
from sklearn.ensemble import GradientBoostingRegressor

model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    loss="squared_error",
    random_state=42
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
```

**Important parameters:**

- `n_estimators`: Number of boosting stages.
- `learning_rate`: Contribution of each tree.
- `max_depth`: Maximum depth of individual trees.
- `loss`: Loss function.
- `subsample`: Fraction of samples used for fitting each tree.

**When to use:**

- When relationships are nonlinear.
- When you want a strong boosting-based model.
- When simpler models do not provide sufficient accuracy.

---

# 11. Histogram Gradient Boosting

**Idea:** Histogram Gradient Boosting uses histogram-based feature binning to train gradient-boosted decision trees efficiently.

It can be faster on larger datasets.

**Implementation:**

```python
from sklearn.ensemble import HistGradientBoostingRegressor

model = HistGradientBoostingRegressor(
    max_iter=200,
    learning_rate=0.05,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
```

**Important parameters:**

- `max_iter`: Number of boosting iterations.
- `learning_rate`: Learning rate.
- `max_leaf_nodes`: Maximum number of leaves.
- `l2_regularization`: L2 regularization strength.
- `max_depth`: Maximum tree depth.

**When to use:**

- When you have a relatively large dataset.
- When you want faster gradient boosting.
- When you want to experiment with histogram-based learning.

**Important note:** This model requires dense input. If your `ColumnTransformer` produces sparse data, set `sparse_output=False` in `OneHotEncoder` or convert the processed matrices to dense arrays if memory permits.

---

# 12. XGBoost Regression

**External library**

**Install:**

```bash
pip install xgboost
```

**Idea:** XGBoost is an optimized gradient boosting algorithm that uses regularization and efficient tree construction.

**Implementation:**

```python
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
```

**Important parameters:**

- `n_estimators`: Number of trees.
- `learning_rate`: Learning rate.
- `max_depth`: Maximum tree depth.
- `subsample`: Fraction of training samples used.
- `colsample_bytree`: Fraction of features used per tree.
- `reg_alpha`: L1 regularization.
- `reg_lambda`: L2 regularization.

**When to use:**

- When you want high predictive performance.
- When working with structured/tabular data.
- When simpler ensemble models are not sufficient.

---

# 13. LightGBM Regression

**External library**

**Install:**

```bash
pip install lightgbm
```

**Idea:** LightGBM is a gradient boosting framework that uses leaf-wise tree growth and histogram-based learning.

It is designed for efficient training on large datasets.

**Implementation:**

```python
from lightgbm import LGBMRegressor

model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
```

**Important parameters:**

- `n_estimators`: Number of boosting iterations.
- `learning_rate`: Learning rate.
- `max_depth`: Maximum tree depth.
- `num_leaves`: Maximum number of leaves.
- `min_child_samples`: Minimum samples in a leaf.
- `reg_alpha`: L1 regularization.
- `reg_lambda`: L2 regularization.

**When to use:**

- When working with large tabular datasets.
- When training speed is important.
- When you want an efficient boosting algorithm.

---

# 14. CatBoost Regression

**External library**

**Install:**

```bash
pip install catboost
```

**Idea:** CatBoost is a gradient boosting algorithm that handles categorical features natively and uses ordered boosting techniques.

Since your categorical features are already one-hot encoded, this implementation will work with your existing processed data.

**Implementation:**

```python
from catboost import CatBoostRegressor

model = CatBoostRegressor(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    loss_function="RMSE",
    verbose=0,
    random_seed=42
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
```

**Important parameters:**

- `iterations`: Number of boosting iterations.
- `learning_rate`: Learning rate.
- `depth`: Depth of the trees.
- `loss_function`: Loss function.
- `l2_leaf_reg`: L2 regularization coefficient.
- `random_seed`: Random seed.

**When to use:**

- When working with categorical and numerical features.
- When you want strong tabular-data performance.
- When you want to experiment with categorical feature handling.

**Important note:** CatBoost is often better tested with original categorical columns rather than one-hot encoding.

---

# 15. Support Vector Regression (SVR)

**Idea:** SVR finds a function that predicts values while allowing errors within a specified margin.

It uses support vectors to determine the regression function.

**Implementation:**

```python
from sklearn.svm import SVR

model = SVR(
    kernel="rbf",
    C=100,
    epsilon=0.1,
    gamma="scale"
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
```

**Important parameters:**

- `kernel`: Kernel function (`linear`, `poly`, `rbf`, `sigmoid`).
- `C`: Regularization parameter.
- `epsilon`: Error margin.
- `gamma`: Kernel coefficient.

**When to use:**

- When the dataset is relatively small or medium-sized.
- When nonlinear relationships exist.
- When you want to experiment with kernel-based regression.

**Important:** SVR is sensitive to feature scaling and target magnitude. Your features are scaled, but consider scaling `y_train` too, then inverse-transform predictions. SVR can also be slow on larger datasets.

---

# 16. K-Nearest Neighbors Regression

**Idea:** KNN Regression finds the K most similar training observations and predicts using their average target value.

It is based on the similarity or distance between observations.

**Implementation:**

```python
from sklearn.neighbors import KNeighborsRegressor

model = KNeighborsRegressor(
    n_neighbors=5,
    weights="distance",
    metric="minkowski",
    p=2
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
```

**Important parameters:**

- `n_neighbors`: Number of nearest neighbors.
- `weights`: Weighting method (`uniform` or `distance`).
- `metric`: Distance metric.
- `p`: Power parameter for Minkowski distance.

**When to use:**

- When similar observations have similar target values.
- When you want a simple instance-based learning algorithm.
- When nonlinear relationships exist.

---

# 17. Gaussian Process Regression

**Idea:** Gaussian Process Regression predicts values using probability distributions and can estimate prediction uncertainty.

It is a probabilistic regression method.

**Implementation:**

```python
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF,
    ConstantKernel,
    WhiteKernel
)

kernel = (
    ConstantKernel(1.0) *
    RBF(length_scale=1.0) +
    WhiteKernel(noise_level=0.1)
)

model = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=2,
    random_state=42
)

model.fit(X_train_processed, y_train)

y_pred, y_std = model.predict(
    X_test_processed,
    return_std=True
)
```

**Important parameters:**

- `kernel`: Kernel function.
- `alpha`: Noise level or regularization.
- `normalize_y`: Whether to normalize the target.
- `n_restarts_optimizer`: Number of optimizer restarts.
- `random_state`: Random seed.

**When to use:**

- When you need uncertainty estimates.
- When working with relatively small datasets.
- When you want probabilistic predictions.

**Important limitation:** Gaussian Process Regression has roughly cubic training complexity with the number of training observations. Your 3,680 training rows and one-hot encoded features make this a poor fit for a straightforward full-dataset experiment. Start with a smaller representative training sample.

---

# Special Regression Models

# 18. Bayesian Ridge Regression

**Idea:** Bayesian Ridge uses Bayesian probability to estimate regression coefficients and account for uncertainty.

It places probability distributions over model parameters.

**Implementation:**

```python
from sklearn.linear_model import BayesianRidge

model = BayesianRidge(
    max_iter=300,
    tol=1e-4
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
```

**Important parameters:**

- `max_iter`: Maximum number of iterations.
- `tol`: Convergence tolerance.
- `alpha_1`: Hyperparameter for the noise precision prior.
- `lambda_1`: Hyperparameter for the coefficient precision prior.
- `compute_score`: Whether to calculate log marginal likelihood.

**When to use:**

- When you want Bayesian regression.
- When uncertainty in coefficients is important.
- When you want an alternative to standard Ridge Regression.

---

# 19. Huber Regression

**Idea:** Huber Regression uses a loss function that is less sensitive to extreme errors than ordinary squared-error regression.

It combines squared loss for smaller errors with absolute loss for larger errors.

**Implementation:**

```python
from sklearn.linear_model import HuberRegressor

model = HuberRegressor(
    epsilon=1.35,
    alpha=0.0001,
    max_iter=500
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
```

**Important parameters:**

- `epsilon`: Controls the threshold for identifying outliers.
- `alpha`: Regularization strength.
- `max_iter`: Maximum number of iterations.
- `tol`: Convergence tolerance.

**When to use:**

- When your dataset contains outliers.
- When ordinary Linear Regression is heavily affected by extreme observations.
- When you want a robust regression model.

---

# 20. RANSAC Regression

**Idea:** RANSAC Regression fits a model using randomly selected subsets of data while trying to identify and ignore outliers.

It is useful when a dataset contains many unusual observations.

**Implementation:**

```python
from sklearn.linear_model import RANSACRegressor
from sklearn.linear_model import LinearRegression

model = RANSACRegressor(
    estimator=LinearRegression(),
    min_samples=0.5,
    max_trials=100,
    random_state=42
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
```

**Important parameters:**

- `estimator`: Base regression model.
- `min_samples`: Minimum number of samples used to fit each model.
- `max_trials`: Maximum number of random iterations.
- `residual_threshold`: Threshold for identifying inliers.
- `random_state`: Random seed.

**When to use:**

- When the dataset contains substantial outliers.
- When you want robust model fitting.
- When ordinary regression is strongly affected by unusual observations.

---

# 21. Theil-Sen Regression

**Idea:** Theil-Sen estimates regression coefficients using multiple subsets of observations, making it more robust to outliers than ordinary Linear Regression.

**Implementation:**

```python
from sklearn.linear_model import TheilSenRegressor

model = TheilSenRegressor(
    max_subpopulation=1000,
    random_state=42
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
```

**Important parameters:**

- `max_subpopulation`: Maximum number of combinations considered.
- `n_subsamples`: Number of samples used for each fit.
- `max_iter`: Maximum number of iterations.
- `tol`: Convergence tolerance.

**When to use:**

- When outliers are present.
- When you want robust linear regression.
- When you want to compare robust regression techniques.

---

# 22. Quantile Regression

**Idea:** Quantile Regression predicts a selected percentile rather than only predicting the average target value.

It can estimate different parts of the conditional target distribution.

**Implementation:**

```python
from sklearn.linear_model import QuantileRegressor

model = QuantileRegressor(
    quantile=0.5,
    alpha=0.1,
    solver="highs"
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
```

**Important parameters:**

- `quantile`: Desired percentile.
- `alpha`: Regularization strength.
- `solver`: Optimization algorithm.

**You can experiment with:**

- `quantile=0.25`: Lower predicted price.
- `quantile=0.50`: Median predicted price.
- `quantile=0.75`: Upper predicted price.

**When to use:**

- When you want to predict different price percentiles.
- When the target distribution is asymmetric.
- When average predictions are not sufficient.

**Important note:** These are conditional quantile predictions, not guaranteed price bounds.

---

# 23. Poisson Regression

**Idea:** Poisson Regression models non-negative count-like target variables using a Poisson distribution.

It is a Generalized Linear Model (GLM).

**Implementation:**

```python
from sklearn.linear_model import PoissonRegressor

model = PoissonRegressor(
    alpha=1.0,
    max_iter=500
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
```

**Important parameters:**

- `alpha`: Regularization strength.
- `max_iter`: Maximum number of iterations.
- `tol`: Convergence tolerance.

**When to use:**

- When the target represents event counts.
- When predicting the number of occurrences.
- When the target is non-negative count data.

**Important:** Your house price target is continuous and not count data. Poisson Regression is not a suitable default for this dataset. Use it to understand the algorithm on appropriate count data.

---

# 24. Gamma Regression

**Idea:** Gamma Regression models positive, continuous, right-skewed target values using a Gamma distribution.

It is useful for modeling positive values with unequal variance.

**Implementation:**

```python
from sklearn.linear_model import GammaRegressor

model = GammaRegressor(
    alpha=1.0,
    max_iter=500
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
```

**Important parameters:**

- `alpha`: Regularization strength.
- `max_iter`: Maximum number of iterations.
- `tol`: Convergence tolerance.
- `fit_intercept`: Whether to calculate the intercept.

**When to use:**

- When the target is strictly positive.
- When the target distribution is right-skewed.
- When modeling positive continuous quantities.

**Important:** House prices are positive, so this is a possible model to experiment with, provided the target contains no zero or negative values.

---

# 25. Tweedie Regression

**Idea:** Tweedie Regression uses a Tweedie probability distribution to model different types of target data depending on its power parameter.

It belongs to the Generalized Linear Model family.

**Implementation:**

```python
from sklearn.linear_model import TweedieRegressor

model = TweedieRegressor(
    power=1.5,
    alpha=1.0,
    link="log",
    max_iter=500
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
```

**Important parameters:**

- `power`: Controls the Tweedie distribution.
- `alpha`: Regularization strength.
- `link`: Link function.
- `max_iter`: Maximum number of iterations.
- `tol`: Convergence tolerance.

**When to use:**

- When modeling non-negative continuous or count-like data.
- When the target distribution has a suitable Tweedie structure.
- When you want to experiment with Generalized Linear Models.

For `power=1.5`, the model uses a compound Poisson-Gamma distribution. Positive target values are required with the log link.

---

# Common Evaluation Code

Use this after each model implementation.

**Evaluation metrics:**

1. MAE – Mean Absolute Error
2. MSE – Mean Squared Error
3. RMSE – Root Mean Squared Error
4. R² – R-squared Score

**Implementation:**

```python
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("MAE:", mean_absolute_error(y_test, y_pred))

mse = mean_squared_error(y_test, y_pred)

print("MSE:", mse)

print("RMSE:", np.sqrt(mse))

print("R2 Score:", r2_score(y_test, y_pred))
```

## Understanding Evaluation Metrics

| Metric | Meaning | Better value |
|---|---|---|
| MAE | Average absolute prediction error | Lower |
| MSE | Average squared prediction error | Lower |
| RMSE | Square root of MSE | Lower |
| R² | Proportion of target variance explained | Higher |

**Note:** MAE, MSE and RMSE depend on the target's scale. R² can be negative when the model performs worse than predicting the test-set mean.

---

# Important Corrections for Your Experiments

1. Keep the same test set for every model.
2. Use scaled features for Ridge, Lasso, Elastic Net, SVR, KNN and Gaussian Process.
3. Tree-based models generally do not require scaling.
4. Gaussian Process is computationally expensive for your dataset.
5. CatBoost is often better tested with original categorical columns rather than one-hot encoding.
6. Poisson Regression is not designed for house prices.
7. Gamma Regression requires strictly positive target values.
8. Polynomial Regression can generate a large number of features, so avoid unnecessarily high degrees.
9. Use cross-validation and hyperparameter tuning to find better model settings.
10. Fit preprocessing transformations only on the training data to avoid data leakage.

---

# Complete Regression Model List

| No. | Regression Model | Category |
|---:|---|---|
| 1 | Linear Regression | Linear |
| 2 | Polynomial Regression | Nonlinear |
| 3 | Ridge Regression | Regularized Linear |
| 4 | Lasso Regression | Regularized Linear |
| 5 | Elastic Net Regression | Regularized Linear |
| 6 | Decision Tree Regression | Tree-based |
| 7 | Random Forest Regression | Ensemble |
| 8 | Extra Trees Regression | Ensemble |
| 9 | AdaBoost Regression | Boosting |
| 10 | Gradient Boosting Regression | Boosting |
| 11 | Histogram Gradient Boosting | Boosting |
| 12 | XGBoost Regression | Boosting |
| 13 | LightGBM Regression | Boosting |
| 14 | CatBoost Regression | Boosting |
| 15 | Support Vector Regression | Kernel-based |
| 16 | K-Nearest Neighbors Regression | Distance-based |
| 17 | Gaussian Process Regression | Probabilistic |
| 18 | Bayesian Ridge Regression | Bayesian |
| 19 | Huber Regression | Robust |
| 20 | RANSAC Regression | Robust |
| 21 | Theil-Sen Regression | Robust |
| 22 | Quantile Regression | Quantile-based |
| 23 | Poisson Regression | Generalized Linear Model |
| 24 | Gamma Regression | Generalized Linear Model |
| 25 | Tweedie Regression | Generalized Linear Model |

**Total: 25 regression model implementations.**

---

# Top Models to Compare First

1. Linear Regression – Baseline model.
2. Ridge Regression – Regularized linear model.
3. Lasso Regression – Feature selection.
4. Elastic Net – Combined regularization.
5. Polynomial Regression – Nonlinear feature relationships.
6. Decision Tree – Nonlinear tree model.
7. Random Forest – Ensemble model.
8. Gradient Boosting – Boosting model.
9. XGBoost – Advanced boosting model.
10. LightGBM – Efficient boosting model.
11. CatBoost – Categorical-data-friendly boosting model.
12. HistGradientBoosting – Efficient histogram-based boosting.

Compare all models using the same test set and the same evaluation metrics.

**Final goal:** Identify the model that provides the best generalization performance, not simply the highest training accuracy.

In [12]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Initialize model
model = LinearRegression()

# Train
model.fit(X_train_processed, y_train)

# Predict
y_pred = model.predict(X_test_processed)

# Evaluate
print("MAE:", mean_absolute_error(y_test, y_pred))

mse = mean_squared_error(y_test, y_pred)
print("MSE:", mse)

print("RMSE:", np.sqrt(mse))

print("R2 Score:", r2_score(y_test, y_pred))

MAE: 158678.30847412633
MSE: 966535591176.4473
RMSE: 983125.4198607863
R2 Score: 0.05227331492503262


# Evaluation Matrix For Regression Training

In [13]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
    mean_squared_log_error,
    median_absolute_error,
    max_error,
    explained_variance_score
)

# Assume y_test and y_pred already exist

# 1. MAE
mae = mean_absolute_error(y_test, y_pred)

# 2. MSE
mse = mean_squared_error(y_test, y_pred)

# 3. RMSE
rmse = root_mean_squared_error(y_test, y_pred)

# 4. R2 Score
r2 = r2_score(y_test, y_pred)

# 5. Adjusted R2
n = len(y_test)
p = X_test_processed.shape[1]

adjusted_r2 = (
    1 - (1 - r2) * (n - 1) / (n - p - 1)
    if n > p + 1 else np.nan
)

# 6. MAPE
mape = mean_absolute_percentage_error(
    y_test, y_pred
) * 100

# 7. MSLE
msle = np.nan

if np.all(y_test >= 0) and np.all(y_pred >= 0):
    msle = mean_squared_log_error(y_test, y_pred)

# 8. Median Absolute Error
median_ae = median_absolute_error(y_test, y_pred)

# 9. Maximum Error
max_err = max_error(y_test, y_pred)

# 10. Explained Variance
explained_variance = explained_variance_score(
    y_test, y_pred
)

# Display results
print("REGRESSION MODEL EVALUATION")
print("---------------------------")

print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)
print("Adjusted R2:", adjusted_r2)
print("MAPE:", mape, "%")
print("MSLE:", msle)
print("Median Absolute Error:", median_ae)
print("Maximum Error:", max_err)
print("Explained Variance:", explained_variance)


REGRESSION MODEL EVALUATION
---------------------------
MAE: 158678.30847412633
MSE: 966535591176.4473
RMSE: 983125.4198607863
R2 Score: 0.05227331492503262
Adjusted R2: -0.1038793708287642
MAPE: 4.381867781640033e+21 %
MSLE: 1.953198873928384
Median Absolute Error: 75495.34912517414
Maximum Error: 26445668.59896614
Explained Variance: 0.05351337240973586


# **Regression algorithms**

## **1. Linear regression algorithms**

| Algorithm             | Type of model                     | Where is it used?                                    | When should you use it?                                                          | How does it work? (Simple idea)                                                                               |
| --------------------- | --------------------------------- | ---------------------------------------------------- | -------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------- |
| Linear Regression     | Linear model                      | House prices, salary prediction, sales prediction    | When an increase in input usually causes a steady increase or decrease in output | Draws the best-fitting straight line between the input and output values.                                     |
| Ridge Regression      | Linear model                      | House prices, financial predictions                  | When many input columns are related to each other                                | Finds the best-fitting line but avoids giving too much importance to any one input column.                    |
| Lasso Regression      | Linear model                      | Sales prediction, feature selection                  | When your dataset has many input columns and some are not useful                 | Finds the best-fitting line and removes less useful input columns by giving them zero importance.             |
| ElasticNet            | Linear model                      | Financial prediction, medical data analysis          | When many input columns are related and some are not useful                      | Combines Ridge and Lasso. It reduces the importance of some inputs and completely removes others.             |
| Polynomial Regression | Linear model with curved features | Population growth, business growth, price prediction | When the output follows a curved pattern instead of a straight line              | Creates additional input columns using squares, cubes, etc., to fit a curved line.                            |
| SGD Regressor         | Linear model                      | Large datasets, online prediction                    | When there is a huge amount of data and training needs to happen step by step    | Starts with an initial prediction line and keeps adjusting it little by little to reduce prediction mistakes. |

## **2. Tree-based algorithms**

| Algorithm               | Type of model             | Where is it used?                                 | When should you use it?                                    | How does it work? (Simple idea)                                                                |
| ----------------------- | ------------------------- | ------------------------------------------------- | ---------------------------------------------------------- | ---------------------------------------------------------------------------------------------- |
| Decision Tree Regressor | Tree-based model          | House prices, customer spending, sales prediction | When the output depends on different conditions or rules   | Asks a series of questions about the input data and predicts the output based on the answers.  |
| Random Forest Regressor | Tree-based ensemble model | House prices, sales, business predictions         | When one decision tree is not giving good results          | Creates many decision trees, collects their predictions, and takes their average.              |
| Extra Trees Regressor   | Tree-based ensemble model | Price prediction, financial prediction            | When you want to use many highly randomized decision trees | Creates many decision trees using more random splitting points and combines their predictions. |

## **3. Boosting algorithms**

| Algorithm                      | Type of model  | Where is it used?                      | When should you use it?                                        | How does it work? (Simple idea)                                                                                          |
| ------------------------------ | -------------- | -------------------------------------- | -------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------ |
| AdaBoost Regressor             | Boosting model | Sales prediction, business forecasting | When simple models are not giving accurate predictions         | Builds multiple small models. Each new model gives more attention to the data points where earlier models made mistakes. |
| Gradient Boosting Regressor    | Boosting model | House prices, sales, customer spending | When the relationship between inputs and output is complicated | Builds models one after another. Each new model tries to correct the mistakes made by the previous models.               |
| HistGradientBoosting Regressor | Boosting model | Large datasets, business forecasting   | When you have a large dataset and want faster training         | Groups similar input values together and builds decision trees using those groups to make predictions faster.            |

## **4. Distance-based and kernel-based algorithms**

| Algorithm                       | Type of model           | Where is it used?                            | When should you use it?                                                             | How does it work? (Simple idea)                                                                                      |
| ------------------------------- | ----------------------- | -------------------------------------------- | ----------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------- |
| K-Neighbors Regressor (KNN)     | Distance-based model    | House prices, product pricing                | When similar things usually have similar prices or values                           | Finds the closest data points and takes their average output to predict the new value.                               |
| Support Vector Regression (SVR) | Support Vector model    | Financial prediction, sales forecasting      | When you have a small or medium dataset and want to control prediction mistakes     | Tries to draw a line or curve that keeps most prediction mistakes within an acceptable limit.                        |
| Kernel Ridge Regression         | Kernel-based model      | Scientific predictions, price prediction     | When the relationship is curved and you also want to control model complexity       | Combines Ridge Regression with a mathematical trick that helps it learn curved relationships.                        |
| Gaussian Process Regressor      | Probability-based model | Weather measurements, scientific experiments | When you want predictions along with an idea of how uncertain those predictions are | Uses existing data to predict possible output values and also estimates how confident it is about those predictions. |

## **5. Neural network algorithms**

| Algorithm     | Type of model        | Where is it used?                                              | When should you use it?                                                           | How does it work? (Simple idea)                                                                                              |
| ------------- | -------------------- | -------------------------------------------------------------- | --------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------- |
| MLP Regressor | Neural network model | Energy prediction, sales forecasting, complex price prediction | When input and output relationships are very complicated and you have enough data | Works like a network of connected units. It makes predictions and repeatedly adjusts its internal values to reduce mistakes. |

## **6. Special regression algorithms**

| Algorithm           | Type of model                   | Where is it used?                                      | When should you use it?                                                                              | How does it work? (Simple idea)                                                                                             |
| ------------------- | ------------------------------- | ------------------------------------------------------ | ---------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------------- |
| Huber Regressor     | Outlier-resistant linear model  | Financial data, price prediction                       | When a few data points have unusually high or low values                                             | Tries to reduce the effect of unusual data points while finding the best-fitting line.                                      |
| RANSAC Regressor    | Outlier-resistant model         | Sensor data, scientific measurements                   | When your dataset contains many incorrect or unusual values                                          | Picks different small groups of data, builds models, and keeps the model that fits the largest group of normal data points. |
| Theil-Sen Regressor | Outlier-resistant linear model  | Scientific measurements, noisy data                    | When you want a straight-line model that is less affected by unusual values                          | Creates several possible lines and uses the middle coefficient values to find a final line.                                 |
| Quantile Regressor  | Linear model                    | Salary prediction, insurance prediction                | When you want to predict the lowest, middle, or highest expected value instead of just the average   | Predicts a particular position in the data, such as the middle value or the 90th percentile.                                |
| Poisson Regressor   | Count prediction model          | Number of accidents, customer visits, number of orders | When your output represents a count, such as 5 accidents or 20 orders                                | Learns patterns from input data to predict how many times something may happen.                                             |
| Gamma Regressor     | Positive-value prediction model | Insurance costs, medical expenses, waiting times       | When your output is always positive and often contains many small values and a few very large values | Learns patterns to predict positive values such as costs, expenses, and waiting times.                                      |

# **Quick guide: Which algorithm should I choose?**

This is the most important table to remember when working on real-world projects.

| If your data looks like this...                         | Try these algorithms  |
| ------------------------------------------------------- | --------------------- |
| Input and output follow a straight-line pattern         | Linear Regression     |
| Many input columns are closely related                  | Ridge Regression      |
| Many input columns are unnecessary                      | Lasso Regression      |
| Some input columns are related and some are unnecessary | ElasticNet            |
| Input and output follow a curved pattern                | Polynomial Regression |
| Output depends on different conditions                  | Decision Tree         |
| One decision tree is not giving good results            | Random Forest         |
| You want more random decision trees                     | Extra Trees           |
| You want each new model to correct earlier mistakes     | Gradient Boosting     |
| You have a large dataset and need faster boosting       | HistGradientBoosting  |
| You want to improve simple models step by step          | AdaBoost              |
| Similar data points have similar output values          | KNN                   |
| You want to control the amount of prediction error      | SVR                   |
| You want predictions along with confidence estimates    | Gaussian Process      |
| Your data has many unusual values                       | Huber or RANSAC       |
| You want to predict a particular percentage or position | Quantile Regression   |
| Your output represents a count                          | Poisson Regression    |
| Your output contains only positive values               | Gamma Regression      |
| You have complex patterns and enough training data      | MLP Regressor         |

### The common algorithms and characteristics

| If your data looks like this... | Algorithms to try |
| --- | --- |
| Input and output follow a straight-line pattern | Linear Regression, Ridge, Lasso |
| Many input columns are closely related (multicollinearity) | Ridge Regression, ElasticNet |
| Many input columns are unnecessary or irrelevant | Lasso Regression, ElasticNet |
| Some input columns are related and some are unnecessary | ElasticNet |
| Input and output follow a curved or nonlinear pattern | Polynomial Regression, Random Forest, SVR |
| Output depends on complex conditions or feature interactions | Decision Tree, Random Forest, Gradient Boosting |
| One decision tree is not giving good results | Random Forest, Extra Trees |
| You want more randomized decision trees | Extra Trees |
| You want each new model to correct earlier mistakes | Gradient Boosting, AdaBoost |
| You have a large tabular dataset | HistGradientBoosting, Random Forest |
| You have a large dataset and need faster boosting | HistGradientBoosting |
| You have a small dataset | Linear Regression, SVR, Gaussian Process |
| Similar data points have similar output values | KNN |
| You want to control the amount of prediction error | SVR |
| You want predictions along with uncertainty estimates | Gaussian Process |
| Your data has many outliers or unusual values | Huber Regressor, RANSAC, Theil-Sen |
| You want to predict a particular percentage or percentile | Quantile Regression |
| Your output represents count data | Poisson Regression |
| Your output contains only positive continuous values | Gamma Regression |
| You have complex patterns and enough training data | MLP Regressor |
| You have a very large dataset and need online or incremental learning | SGD Regressor |

### Simple and Quick to remember

| Your dataset characteristic               | Algorithms to try first                   |
| ----------------------------------------- | ----------------------------------------- |
| Linear relationship                       | Linear Regression, Ridge, Lasso           |
| Many correlated features                  | Ridge, ElasticNet                         |
| Many irrelevant features                  | Lasso, ElasticNet                         |
| Nonlinear relationship                    | Polynomial Regression, Random Forest, SVR |
| Complex feature interactions              | Random Forest, Gradient Boosting          |
| Large tabular dataset                     | HistGradientBoosting, Random Forest       |
| Small dataset                             | Linear Regression, SVR, Gaussian Process  |
| Many outliers                             | Huber, RANSAC, Theil-Sen                  |
| Similar observations have similar targets | KNN                                       |
| Need prediction uncertainty               | Gaussian Process                          |
| Need percentile predictions               | Quantile Regression                       |
| Very large dataset with online learning   | SGD Regressor                             |

# **Regression evaluation metrics**

**Unlike classification, regression predicts continuous numerical values. Therefore, we use different evaluation metrics to measure how close the model's predictions are to the actual values.**

| Metric                                    | Simple explanation                                                          | What does it tell us?                                                    |
| ----------------------------------------- | --------------------------------------------------------------------------- | ------------------------------------------------------------------------ |
| **MAE (Mean Absolute Error)**             | The average absolute difference between actual and predicted values.        | How far the predictions are from the actual values on average.           |
| **MSE (Mean Squared Error)**              | The average of squared differences between actual and predicted values.     | How much prediction errors are, giving more importance to larger errors. |
| **RMSE (Root Mean Squared Error)**        | The square root of MSE.                                                     | The typical prediction error in the same unit as the target variable.    |
| **R² Score (R-Squared)**                  | Measures how well the model explains the variation in the actual values.    | How well the model fits the data.                                        |
| **Adjusted R²**                           | Measures model performance while considering the number of predictors used. | Whether adding more features actually improves the model.                |
| **MAPE (Mean Absolute Percentage Error)** | Measures the average prediction error as a percentage.                      | How large the prediction errors are in percentage terms.                 |

**Remember:**

- **MAE:** Average absolute error.
- **MSE:** Penalizes larger errors more heavily.
- **RMSE:** Error in the original unit.
- **R²:** How well the model explains the data.
- **Adjusted R²:** Considers the number of features.
- **MAPE:** Error expressed as a percentage.

## **Important differences to remember**

Here are some concepts that are particularly important when comparing algorithms.

**1. Linear Regression vs Ridge vs Lasso**

- Linear Regression: Minimizes squared prediction errors.
- Ridge: Minimizes squared errors while penalizing large coefficients.
- Lasso: Minimizes squared errors while encouraging some coefficients to become zero.

**2. Decision Tree vs Random Forest vs Gradient Boosting**

**Decision Tree**

One tree makes predictions using decision rules. check:
[medium.com](https://medium.com/%40ricardobgarcia/um-olhar-sobre-o-comportamento-n%C3%A3o-linear-de-%C3%A1rvores-de-decis%C3%A3o-e-florestas-aleat%C3%B3rias-9473a5e2ee9d?utm_source=chatgpt.com)

**Random Forest**

Multiple independently trained randomized trees contribute to an averaged prediction. check:
[datacamp.com](https://www.datacamp.com/es/tutorial/random-forest-regression?utm_source=chatgpt.com)

**Gradient Boosting**

Trees are added sequentially to improve the predictions of the existing ensemble. check:
[medium.com](https://medium.com/%40abhaysingh71711/gradient-boosting-explained-turning-weak-models-into-winners-c5d145dca9ab?utm_source=chatgpt.com)

**A few implementation considerations:**

- Scale features for distance-based, kernel-based, neural network and regularized linear models.
- Use cross-validation to compare models reliably.
- Avoid data leakage by fitting preprocessing only on training folds.
- Tune hyperparameters before making your final model selection.
- Don't assume the highest training R² means the best model.

## **Official documentation**

For further reference:

- [Scikit-learn – Linear Models](https://sklearn.org/stable/modules/linear_model.html?utm_source=chatgpt.com).
- [Scikit-learn – Decision Trees](https://scikit-learn.org/stable/modules/tree?utm_source=chatgpt.com).
- [Scikit-learn – Ensemble Methods](https://scikit-learn.org/stable/modules/ensemble?utm_source=chatgpt.com).
- [Scikit-learn – Supervised Learning Algorithms](https://scikit-learn.org/stable/supervised_learning.html?utm_source=chatgpt.com).